### Environment Setup

Same kernel and dependency pins as `day-metrics.ipynb`. Run from the repo root. Public market data only; no API keys, no orders. Live trading stays off at all times in this notebook.

# Day Trader Controls

In its nascent stage, the goal of this trading bot is learning and testing rigging. Fees, day-trade rules, and spreads are key concerns. Its architecture design is mostly conservative, focusing on edge validation in out-of-sample data, and testing operational pipelines end to end. This means monitoring the model to avoid chasing small returns that are eaten by market fees. Test results from `day-metrics.ipnyb` adds to this concern, as inital runs yield limited edge in out-of-sample 

This design follows a selective, high-conviction swing strategy plan. This is not a scalper. A trade held for a clean three or five percent move does not care about a 0.2 percent round trip; a trade chasing half a percent is eaten by it, this is our floor.

The minimum-edge floor is enforced to manage high-volume trading where a model of cumulative return appears busier by taking many small trades, while fees quietly eat the account. We propose three guardrails against this: i) a cap limit on number of trades-per-day, ii) minimum-edge floor limit on thinner trades, iii) and after-fee validation protocol. Effectively, this restricts the model from high frequency trades on thinner margins with fees reported. 


### 1. Key Variables

Every tunable parameter below is stored in one `config.py` helper never hard-coded, and the risk-bounding
ones are operator-owned and held outside the model's reach.

**Selection screen**
- `quote_volume_24h_usdt` — 24h quote volume, USDT. Hard gate floor
- `atr_pct` — ATR(14) ÷ current price, as a percent. Band, two edges, floor and ceiling
- `spread_pct` — (ask − bid) ÷ mid, as a percent. Hard gate, ceiling
- `candle_count` — minimum number of daily candles returned
- `pass` — boolean, true only if all gates clear

**Edge and exit**
- `round_trip_fee_pct` — venue round-trip cost (Binance ≈ 0.15–0.20%)
- `expected_slippage_pct` — modelled slippage per round trip
- `edge_floor_pct` — minimum net expected move to allow a trade.
- `take_profit_pct` — target gain; must exceed `edge_floor_pct`
- `stop_loss_pct` — max loss per trade before forced exit
- `net_expected_move = est_move_pct − round_trip_fee_pct − expected_slippage_pct`

**Position and frequency limits**
- `max_trades_per_day` — hard cap.
- `max_open_positions` — concurrent positions (3–4 at current account size).
- `position_size_pct` — capital per trade; volatility-scaled (see s.3)
- `hold_window_days` — range, fitted by walk-forward, not fixed (swing band)

**Signal layers derived in `day-metrics`)**
- `macd`, `macd_signal`, `macd_hist`, `hist_slope`, `converging`
- `cross_up`, `cross_down`, `guarded_buy`, `guarded_sell`, `epsilon` (noise band)
- `bear_div`, `bull_div` (swing-pivot divergence flags)
- four-vote score: `w_macd*MACD + w_ma*MA + w_fib*Fib + w_candle*Candle`
- `threshold` — vote score to fire (2 standard, 3 when few trades to learn from)

**Evaluation harness** 
- `oos_window`, `train_window`, `regime_set`, `fee_assumption` — all fixed across comparisons
- `experiment_log` — one line per walk-forward run: params, OOS after-fee result, kept/discarded


### Policing Day Trades

- **Who sets the floor.** The minimum-edge floor is set by the operator or fixed outside the model's reach, never tuned by the model itself. The model is judged on the results the floor produces, so a model that could lower the floor would lower it to book more wins. The floor is reviewed weekly by the operator, not learned.
- **Defence one — trades-per-day cap.** A hard limit on how many trades can exist in a day, set in code. This alone makes the volume trick mechanically impossible: the model cannot churn because it cannot place the orders.
- **Defence two — minimum-edge floor.** A trade is refused unless its expected net move clears the floor. Net means after round-trip fee and after expected slippage, not gross. On Binance crypto the round trip is roughly 0.2 percent, so the floor sits well above that, in the region of one and a half to two percent expected move.
- **Defence three — out-of-sample validation.** Walk-forward only. Tune weights and threshold on a training segment, score once on an untouched test segment, move forward, repeat. Report only the out-of-sample, after-fee, multi-regime aggregate. No in-sample result is ever the verdict.


### Two venues, two fence sets

- **Binance crypto.** Round-trip cost roughly 0.15 to 0.2 percent (0.075 percent per side paying fees in BNB, 0.1 percent otherwise). Fee is the binding fence. Keep a BNB balance topped up for the discount.
- **Alpaca equities.** Per-trade cost effectively zero (regulatory pass-throughs on sells only, cents per trade). The binding fence is the pattern-day-trader rule: a sub-25k cash account is capped at three day-trades per rolling five days. Margin interest at 6.25 percent annualized applies to any leveraged overnight hold — trade cash-only to avoid it. Confirm the account is direct self-directed, not partner-routed, or the zero-commission assumption breaks.
- The model must know **which fence set governs the order it is about to place.**

In [18]:

import sys, subprocess
from pathlib import Path

# Pin the kernel to the same interpreter the rest of the repo uses.
REQUIRED_PY = (3, 11)
if sys.version_info[:2] != REQUIRED_PY:
    raise RuntimeError(
        f"This notebook needs Python {REQUIRED_PY[0]}.{REQUIRED_PY[1]}, "
        f"but the kernel is {sys.version.split()[0]}. Switch the kernel and re-run.")

# Install the exact versions in inputs/requirements.txt (shared with day-metrics).
REQUIREMENTS = Path("inputs/requirements.txt")
if not REQUIREMENTS.exists():
    raise FileNotFoundError(f"{REQUIREMENTS} not found - run from the repo root.")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--break-system-packages", "--disable-pip-version-check",
    "-r", str(REQUIREMENTS)], check=True)

import warnings, numpy as np, pandas as pd
from datetime import datetime, timezone
import ccxt
import plotly, plotly.graph_objects as go, plotly.io as pio
warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype+notebook_connected"
from IPython.display import HTML, display

def show(fig):
    """Render a plotly figure inline via HTML + CDN.
    .show() with the mimetype renderer can blank out in Positron / VS Code;
    to_html with a CDN plotly.js renders reliably across IDEs (same pattern
    as day-metrics.ipynb)."""
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

OUTPUTS = Path("outputs"); OUTPUTS.mkdir(exist_ok=True)
print(f"Environment ready  ·  python {sys.version.split()[0]}")
print(f"  ccxt {ccxt.__version__} · pandas {pd.__version__} · "
      f"numpy {np.__version__} · plotly {plotly.__version__}")

Environment ready  ·  python 3.11.13
  ccxt 4.5.59 · pandas 2.3.3 · numpy 2.4.6 · plotly 6.8.0


### Operator-Owned Configuration

This realises **§1 Key Variables**. Every tunable lives here in one block, not scattered through the code. The 🔒 entries are the risk-bounding fences: operator-set, reviewed weekly, **held outside the model's reach** (a model judged on the results a fence produces would loosen it to book wins). Secrets stay in `inputs/config.py` (macOS Keychain); this block is strategy only and holds no keys. When the bot is built, lift this dict into a locked module the strategy file cannot write to.

In [19]:

# ── CONFIG ── outside the model's reach.
CONFIG = dict(
    # Venue fences — Binance crypto round trip (0.075%/side with BNB, 0.1% without)
    round_trip_fee_pct    = 0.15,      # two sides, BNB-discounted; use 0.20 without BNB
    expected_slippage_pct = 0.05,      # modelled slippage per round trip

    # Edge & exit (per-trade economics)
    edge_floor_pct        = 1.5,       # minimum NET expected move to allow a trade
    take_profit_pct       = 3.0,       # target gain; must exceed edge_floor_pct
    stop_atr_mult         = 1.5,       # stop distance = this × daily ATR%  (ATR-based stop)

    # ATR band — selection filter AND live guardrail
    atr_length            = 14,
    atr_floor_pct         = 2.5,       # below this a coin can't reach TP in the window
    atr_ceiling_pct       = 12.0,      # above this it gaps through stop/TP unpredictably

    # Hard gates (selection screen)
    min_quote_volume_usdt = 30_000_000,# liquidity floor, 24h quote volume USDT
    max_spread_pct        = 0.05,      # spread ceiling, well under the fee
    min_history           = 120,       # candles required to span regimes
    history_limit         = 180,       # candles pulled per coin (90–180)

    # Frequency & position limits
    max_trades_per_day    = 3,         # hard cap — defence one against volume-hiding
    max_open_positions    = 4,         # concurrent positions (3–4 at this account size)
    risk_per_trade_pct    = 1.0,       # % of account risked per trade (constant $ risk)
    max_position_pct      = 30.0,      # cap any single position at this % of account
    min_notional_usdt     = 10.0,      # venue minimum clip; below this a trade isn't worth it

    # Hold window — a walk-forward RANGE, not a fixed number (swing band)
    hold_window_days_min  = 1,
    hold_window_days_max  = 10,

    # Account & scan
    account_usdt          = 750.0,     # proving-rig size (~$500–1000)
    scan_top_n            = 25,        # scan wide (15–25), hold few (3–4)
)

# Public, read-only Binance client (no keys; never armed for orders here).
EXCHANGE = "binance"
exchange = getattr(ccxt, EXCHANGE)()
exchange.set_sandbox_mode(False)   # public market data only

ALLOW_SYNTHETIC = True   # if a live fetch fails (offline / geo-blocked), fall back to
                         # deterministic synthetic data so the notebook still renders.
print(f"Configured {EXCHANGE} | account ${CONFIG['account_usdt']:.0f} | "
      f"edge floor {CONFIG['edge_floor_pct']}% | ATR band "
      f"{CONFIG['atr_floor_pct']}–{CONFIG['atr_ceiling_pct']}% | scan top {CONFIG['scan_top_n']}")

Configured binance | account $750 | edge floor 1.5% | ATR band 2.5–12.0% | scan top 25


### Data Streams

Three pulls per coin from CCXT, all public: **daily OHLCV** (last 90–180 candles, drop the unclosed bar), the **24h quote volume** from the ticker, and the **order-book top** for the spread. The universe is the most-traded `/USDT` spot pairs, scoped to the one venue we execute on (the manifesto's caution: do not cast across all 205 exchanges). Each fetch degrades to deterministic synthetic data if the network is unavailable, so nothing hard-crashes.

In [20]:

def synthetic_ohlcv(symbol, n=None, daily_vol=None, seed=None):
    """Deterministic fake OHLCV for offline/geo-blocked demo. Seeded by symbol."""
    n = n or CONFIG["history_limit"]
    s = abs(hash(symbol)) % (2**32) if seed is None else seed
    rng = np.random.default_rng(s)
    dv = daily_vol if daily_vol is not None else float(rng.uniform(0.01, 0.10))
    rets = rng.normal(0, dv, n)
    close = 100 * np.exp(np.cumsum(rets))
    high  = close * (1 + np.abs(rng.normal(0, dv/2, n)))
    low   = close * (1 - np.abs(rng.normal(0, dv/2, n)))
    openp = np.r_[close[0], close[:-1]]
    vol   = rng.uniform(1e6, 5e6, n)
    idx   = pd.date_range("2025-01-01", periods=n, freq="D")
    return pd.DataFrame({"open":openp,"high":high,"low":low,"close":close,"volume":vol}, index=idx)

def fetch_daily(symbol, limit=None):
    """Daily OHLCV, unclosed bar dropped. Falls back to synthetic if offline."""
    limit = limit or CONFIG["history_limit"]
    try:
        bars = exchange.fetch_ohlcv(symbol, timeframe="1d", limit=limit)
        df = pd.DataFrame(bars[:-1], columns=["timestamp","open","high","low","close","volume"])
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        return df.set_index("timestamp")
    except Exception as e:
        if ALLOW_SYNTHETIC:
            return synthetic_ohlcv(symbol, n=limit)
        raise

def fetch_quote_volume(symbol):
    """24h quote volume in USDT (the liquidity gate input)."""
    try:
        return float(exchange.fetch_ticker(symbol).get("quoteVolume") or 0.0)
    except Exception:
        if ALLOW_SYNTHETIC:
            return float(np.random.default_rng(abs(hash(symbol))%2**32).uniform(2e6, 2.5e8))
        raise

def fetch_spread_pct(symbol):
    """Top-of-book spread as a percent of mid (the spread gate input)."""
    try:
        ob = exchange.fetch_order_book(symbol, limit=5)
        bid, ask = ob["bids"][0][0], ob["asks"][0][0]
        mid = (bid + ask) / 2
        return (ask - bid) / mid * 100.0
    except Exception:
        if ALLOW_SYNTHETIC:
            return float(np.random.default_rng(abs(hash(symbol))%2**32 + 7).uniform(0.005, 0.12))
        raise

def build_universe(top_n=None):
    """Most-traded /USDT spot pairs by quote volume on the execution venue."""
    top_n = top_n or CONFIG["scan_top_n"]
    try:
        tk = exchange.fetch_tickers()
        usdt = [(s, d.get("quoteVolume") or 0) for s, d in tk.items()
                if s.endswith("/USDT") and ":" not in s]
        uni = [s for s, _ in sorted(usdt, key=lambda x: -x[1])[:top_n]]
        if uni:
            return uni
    except Exception:
        pass
    # offline demo universe: a deliberate spread of regimes for the screen to sort
    print("(offline) using synthetic demo universe")
    return ["BTC/USDT","ETH/USDT","SOL/USDT","BNB/USDT","XRP/USDT","ADA/USDT",
            "AVAX/USDT","LINK/USDT","LTC/USDT","DOGE/USDT","TRX/USDT","DOT/USDT"]

universe = build_universe()
print(f"universe ({len(universe)}):", ", ".join(universe[:12]), "..." if len(universe) > 12 else "")

universe (25): USDC/USDT, BTC/USDT, ETH/USDT, NIGHT/USDT, RE/USDT, USD1/USDT, SOL/USDT, ZEC/USDT, WLD/USDT, XRP/USDT, BNB/USDT, AVAX/USDT ...


### Volatility Filter 

- ATR Band: Average true range across a 14 day period reported as percentage of current price. ATR indicator of typical daily movement.
- Floor: a coin must move enough per day to reach the take-profit inside the hold window. Floor sits above the net edge requirement (i.e. ~2-3% daily ATR).
- Ceiling: above some ATR the coin gaps through stop and take-profit unpredictably and indicators lose meaning. Ceiling sits below where the coin detonates.
- These inform two unrelated operations:
    1. Selection Filter: the gate that admits or rejects a coin from the universe before the model sees it.
    2. Live Guardrail: keeps the model from trading a coin that has drifted out of the tradable band.



In [21]:

def compute_atr_pct(df, length=None):
    """ATR(length) as a percent of price. TR = max(H−L, |H−Cprev|, |L−Cprev|);
    ATR = simple moving average of TR (as written in §2); atr_pct = ATR/price×100."""
    length = length or CONFIG["atr_length"]
    h, l, c = df["high"], df["low"], df["close"]
    pc = c.shift(1)
    tr = pd.concat([(h - l).abs(), (h - pc).abs(), (l - pc).abs()], axis=1).max(axis=1)
    atr = tr.rolling(length).mean()
    return (atr / c) * 100.0

def atr_band_figure(symbol):
    """Live-guardrail view: ATR(14)% over time against the tradable band."""
    df = fetch_daily(symbol)
    atr_pct = compute_atr_pct(df)
    lo, hi = CONFIG["atr_floor_pct"], CONFIG["atr_ceiling_pct"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=atr_pct.index, y=atr_pct, name="ATR(14) %",
                             line=dict(color="#0B3D66", width=1.6)))
    fig.add_hrect(y0=lo, y1=hi, fillcolor="#2E8B57", opacity=0.10, line_width=0,
                  annotation_text="tradable band", annotation_position="top left")
    fig.add_hline(y=lo, line=dict(color="#2E8B57", dash="dash"),
                  annotation_text=f"floor {lo}%")
    fig.add_hline(y=hi, line=dict(color="#B22222", dash="dash"),
                  annotation_text=f"ceiling {hi}%")
    fig.update_layout(title=f"{symbol} — daily ATR% vs tradable band (selection + live guardrail)",
                      template="plotly_white", height=360,
                      yaxis_title="ATR(14) as % of price", xaxis_title=None,
                      margin=dict(l=40, r=20, t=50, b=30))
    return fig

show(atr_band_figure(universe[0]))

### 2. Four-Gate Selection Rule

We propose the following screening assessment to evaluate coins individually on a weekly basis.

Inputs: daily OHLCV, last 90–180 candles, plus current order-book top.

- Gate 1 Liquidity - used to reject below floor before computing anything else.
 - `quote_volume_24h_usdt` from the ticker. 
 
- Gate 2 ATR Band - reported as percent of price showing average true range per day using:  
  - `TR = max(high−low, |high−prev_close|, |low−prev_close|)`
  - `14-period moving average of TR`
  - `atr_pct = ATR / current_price × 100`
  - `atr_floor_pct ≤ atr_pct ≤ atr_ceiling_pct`. 

- Gate 3 Spread - Rejected if above a tight ceiling that signficantly under the fee floor, derived from order book as:
  - `spread_pct = (ask − bid) / mid × 100` 

- Gate 4 History Range -  Rejects young tokens where Ichimoku cloud clearly lacking:
  - `candle_count < min_history`. 

This informs selection based on `{symbol, quote_volume_24h_usdt, atr_pct, spread_pct, candle_count, pass}`.
Targets include a sample of 15–25 coins allowing positions on 3–4 coins.


In [22]:

def screen_coin(symbol, cfg=CONFIG):
    """Run the four gates on one coin. Liquidity is evaluated first (it is the cheap,
    hard gate); all four numbers are recorded for the table either way."""
    df = fetch_daily(symbol, cfg["history_limit"])
    candle_count = int(len(df))
    quote_vol = fetch_quote_volume(symbol)
    spread = fetch_spread_pct(symbol)
    atr_series = compute_atr_pct(df, cfg["atr_length"])
    atr_pct = float(atr_series.iloc[-1]) if atr_series.notna().any() else float("nan")

    g1 = quote_vol >= cfg["min_quote_volume_usdt"]                       # liquidity (hard)
    g2 = (cfg["atr_floor_pct"] <= atr_pct <= cfg["atr_ceiling_pct"]) if atr_pct == atr_pct else False
    g3 = spread <= cfg["max_spread_pct"]                                 # spread (hard)
    g4 = candle_count >= cfg["min_history"]                              # history (sufficiency)
    gates = {"liquidity": bool(g1), "atr_band": bool(g2), "spread": bool(g3), "history": bool(g4)}
    passed = all(gates.values())
    fail = "" if passed else ",".join(k for k, v in gates.items() if not v)
    return {"symbol": symbol,
            "quote_volume_24h_usdt": round(quote_vol, 0),
            "atr_pct": round(atr_pct, 3),
            "spread_pct": round(spread, 4),
            "candle_count": candle_count,
            "pass": bool(passed), "fail_reason": fail, **gates}

def run_screen(universe, cfg=CONFIG, save=True):
    rows = []
    for s in universe:
        try:
            rows.append(screen_coin(s, cfg))
        except Exception as e:
            print("skip", s, type(e).__name__, str(e)[:60])
    tab = pd.DataFrame(rows).sort_values(["pass", "quote_volume_24h_usdt"],
                                         ascending=[False, False]).reset_index(drop=True)
    if save and len(tab):
        stamp = datetime.now(timezone.utc).strftime("%Y%m%d")
        path = OUTPUTS / f"Candidates_{stamp}.csv"
        tab.to_csv(path, index=False)
        print(f"saved {path}  ({int(tab['pass'].sum())} of {len(tab)} passed)")
    return tab

screen = run_screen(universe)
cols = ["symbol","quote_volume_24h_usdt","atr_pct","spread_pct","candle_count","pass","fail_reason"]
screen[cols]

saved outputs/Candidates_20260620.csv  (9 of 25 passed)


,symbol,quote_volume_24h_usdt,atr_pct,spread_pct,candle_count,pass,fail_reason
0,BTC/USDT,886267093.0,3.327,0.0000,179,True,
1,ETH/USDT,281751259.0,4.724,0.0006,179,True,
2,SOL/USDT,132179625.0,5.629,0.0144,179,True,
3,ZEC/USDT,92630321.0,11.183,0.0021,179,True,
4,XRP/USDT,82189772.0,5.077,0.0088,179,True,
5,BNB/USDT,56036232.0,3.500,0.0017,179,True,
6,AVAX/USDT,48339641.0,6.897,0.0169,179,True,
7,TAO/USDT,38010941.0,9.743,0.0438,179,True,
8,NEAR/USDT,36614289.0,10.151,0.0463,179,True,
9,USDC/USDT,905288184.0,0.109,0.0010,179,False,atr_band


The candidate universe is the rows where `pass` is true. The chart below places every scanned coin on the two axes that matter most — liquidity against volatility — with the hard liquidity floor and the ATR band drawn in. Survivors sit inside the green box: liquid enough (above the floor) and lively-but-not-detonating (inside the band).

In [23]:

def screen_scatter(tab, cfg=CONFIG):
    passed = tab[tab["pass"]]; failed = tab[~tab["pass"]]
    fig = go.Figure()
    for d, name, col in [(failed, "rejected", "#B22222"), (passed, "candidate", "#2E8B57")]:
        if len(d):
            fig.add_trace(go.Scatter(
                x=d["atr_pct"], y=d["quote_volume_24h_usdt"], mode="markers+text",
                text=d["symbol"].str.replace("/USDT", ""), textposition="top center",
                textfont=dict(size=9), name=name,
                marker=dict(size=11, color=col, opacity=0.8,
                            line=dict(width=1, color="white"))))
    fig.add_vrect(x0=cfg["atr_floor_pct"], x1=cfg["atr_ceiling_pct"],
                  fillcolor="#2E8B57", opacity=0.07, line_width=0)
    fig.add_vline(x=cfg["atr_floor_pct"], line=dict(color="#2E8B57", dash="dash"))
    fig.add_vline(x=cfg["atr_ceiling_pct"], line=dict(color="#B22222", dash="dash"))
    fig.add_hline(y=cfg["min_quote_volume_usdt"], line=dict(color="#888", dash="dot"),
                  annotation_text="liquidity floor", annotation_position="bottom right")
    fig.update_layout(title="Selection screen — liquidity vs volatility (survivors inside the band, above the floor)",
                      template="plotly_white", height=440, yaxis_type="log",
                      xaxis_title="daily ATR(14) %", yaxis_title="24h quote volume (USDT, log)",
                      margin=dict(l=60, r=20, t=50, b=40))
    return fig

def spread_bars(tab, cfg=CONFIG):
    t = tab.sort_values("spread_pct")
    colors = ["#2E8B57" if v <= cfg["max_spread_pct"] else "#B22222" for v in t["spread_pct"]]
    fig = go.Figure(go.Bar(x=t["symbol"].str.replace("/USDT",""), y=t["spread_pct"],
                           marker_color=colors))
    fig.add_hline(y=cfg["max_spread_pct"], line=dict(color="#888", dash="dash"),
                  annotation_text=f"spread ceiling {cfg['max_spread_pct']}%")
    fig.update_layout(title="Spread gate — top-of-book spread vs ceiling",
                      template="plotly_white", height=320, yaxis_title="spread %",
                      margin=dict(l=50, r=20, t=50, b=40))
    return fig

show(screen_scatter(screen))
show(spread_bars(screen))


### 3. Edge floor, exit, and position sizing (the quantitative core)

**Net edge test (entry gate).** A trade is refused unless:
`net_expected_move = est_move_pct − round_trip_fee_pct − expected_slippage_pct ≥ edge_floor_pct`
Measured on **net**, not gross. This is the fence, not an alarm — it refuses, it does not warn.

**Exit coupling.** Required ordering, enforce in code:
`edge_floor_pct < take_profit_pct` and `atr_floor_pct` large enough that a normal multi-day range
reaches `take_profit_pct` inside `hold_window_days`. `stop_loss_pct` set against the same ATR.

**Volatility-scaled position sizing (industry standard, ATR-based).** Keep dollar risk roughly
constant across coins: size each position so that `stop_loss_pct × position_value ≈ constant risk
budget`. Higher-ATR coin → smaller position; calmer coin → larger. At ~$500–1000 account, floor
position size at the venue's minimum notional, and prefer 3–4 larger positions over many tiny clips
that waste edge on fees and bump minimums.

---

### 4. Hold period — resolved as a range, not a fixed number

The earlier "1–3 days" was a gut figure. Resolution: **swing is the band** (days to ~2 weeks),
scalping and day-trading ruled out (fees and PDT rule), position-trading ruled out (underuses the
signal stack). Within the swing band, make `hold_window_days` a **walk-forward parameter** (e.g.
search 1–10 days) and let out-of-sample results pick it. Shorter holds demand higher `atr_pct`;
longer holds tolerate lower. Same dial, two ends.

---

### 5. Autoresearch lifts (from karpathy/autoresearch, adapted)

The repo's architecture maps onto this project, but its separation is enforced by **instruction**
("do not modify this file"); a model optimised against the metric needs separation enforced by
**code structure**. Lifts:
- **Narrow editable surface.** Define the one file/module the model may tune; lock everything else
  (evaluation, fee floor, ATR band, guardrails) in files the model cannot write to.
- **Experiment log.** Each walk-forward run writes one line: params tried, OOS after-fee result,
  kept or discarded, why. This is the artefact the operator reviews weekly.
- **Frozen yardstick.** Same OOS windows, fees, regime set across every comparison. Any change to
  the harness is a deliberate, separately-recorded human act, never mid-comparison.
- **program.md pattern.** The human-owned instruction layer (this document / PROJECT-BRIEF.md) is
  iterated by the operator; the model iterates the strategy. Mirrors his program.md vs train.py split.
- **Caution:** trading is adversarial and non-stationary; an OOS metric can rot live in a way his
  fixed-corpus val_bpb never does. The weekly human review is not automatable away.

---

### 6. Design parameters (carried forward, unchanged)

#### Three stacked defences against volume-hiding
- **Who sets the floor.** Operator-set or fixed outside the model's reach, never tuned by the model. Reviewed weekly.
- **Defence one — trades-per-day cap.** Hard, in code. Churn becomes mechanically impossible.
- **Defence two — minimum-edge floor.** Refuses trades whose net move does not clear the floor.
- **Defence three — out-of-sample validation.** Walk-forward only; report OOS after-fee multi-regime aggregate.

#### Floor is a fence, not an alarm
- Refuses the trade; optional alarm band may sit above the fence, but the fence protects the account.
- Measured on estimated move minus fee minus slippage — the honest waterline, above the raw fee.

#### Two venues, two fence sets
- **Binance crypto.** Round trip ≈ 0.15–0.20% (0.075%/side with BNB). Fee is the binding fence. Keep BNB topped up.
- **Alpaca equities.** Per-trade ≈ zero (reg pass-throughs, sells only). Binding fence is the PDT rule: sub-25k cash account capped at 3 day-trades / rolling 5 days. Margin interest 6.25%/yr on leveraged overnight holds — trade cash-only. Confirm account is direct self-directed, not partner-routed (else 0%-3% commission band applies).
- Model must know which fence set governs each order.

In [24]:

# ── Net edge fence (entry gate) ──────────────────────────────────────────────
def net_edge(est_move_pct, cfg=CONFIG):
    """NET expected move after round-trip fee and slippage (not gross)."""
    return est_move_pct - cfg["round_trip_fee_pct"] - cfg["expected_slippage_pct"]

def net_edge_ok(est_move_pct, cfg=CONFIG):
    """The fence: refuse unless net move clears the floor. Refuses, does not warn."""
    return net_edge(est_move_pct, cfg) >= cfg["edge_floor_pct"]

# ── Exit coupling (enforced ordering) ────────────────────────────────────────
def validate_exit_coupling(cfg=CONFIG):
    rt = cfg["round_trip_fee_pct"] + cfg["expected_slippage_pct"]
    days_to_tp = cfg["take_profit_pct"] / cfg["atr_floor_pct"]   # at the ATR floor
    return {
        "edge_floor < take_profit":            cfg["edge_floor_pct"] < cfg["take_profit_pct"],
        "round_trip < edge_floor":             rt < cfg["edge_floor_pct"],
        "take_profit > 2x round_trip (alive)":  cfg["take_profit_pct"] > 2 * rt,
        "TP reachable at ATR floor within window":
            days_to_tp <= cfg["hold_window_days_max"],
        f"  ↳ days to TP at {cfg['atr_floor_pct']}%/day ≈ {days_to_tp:.1f}": True,
    }

# ── ATR-scaled position sizing (constant dollar risk) ────────────────────────
def position_plan(atr_pct, cfg=CONFIG, account=None):
    """Size so dollar risk is ~constant across coins: higher ATR -> smaller position.
    stop% = stop_atr_mult × daily ATR%; size = risk_budget / stop%. Capped at
    max_position_pct, floored at the venue minimum notional."""
    account = account or cfg["account_usdt"]
    stop_pct = cfg["stop_atr_mult"] * atr_pct
    risk_budget = account * cfg["risk_per_trade_pct"] / 100.0
    raw = risk_budget / (stop_pct / 100.0) if stop_pct > 0 else 0.0
    capped = min(raw, account * cfg["max_position_pct"] / 100.0)
    final = max(capped, cfg["min_notional_usdt"])
    return {"atr_pct": round(atr_pct, 3), "stop_pct": round(stop_pct, 3),
            "risk_budget_usdt": round(risk_budget, 2), "raw_size_usdt": round(raw, 2),
            "position_usdt": round(final, 2),
            "floored_at_min": bool(final == cfg["min_notional_usdt"] and capped < cfg["min_notional_usdt"])}

print("Net-edge fence (gross est. move -> net -> allowed?):")
for m in (1.0, 1.5, 2.0, 3.5):
    print(f"  est {m:>4}%  ->  net {net_edge(m):>5.2f}%  ->  {'ALLOW' if net_edge_ok(m) else 'refuse'}")

print("\nExit-coupling checks (config must satisfy all):")
for k, v in validate_exit_coupling().items():
    print(f"  [{'ok' if v else 'XX'}] {k}" if not k.strip().startswith('↳') else f"     {k}")

Net-edge fence (gross est. move -> net -> allowed?):
  est  1.0%  ->  net  0.80%  ->  refuse
  est  1.5%  ->  net  1.30%  ->  refuse
  est  2.0%  ->  net  1.80%  ->  ALLOW
  est  3.5%  ->  net  3.30%  ->  ALLOW

Exit-coupling checks (config must satisfy all):
  [ok] edge_floor < take_profit
  [ok] round_trip < edge_floor
  [ok] take_profit > 2x round_trip (alive)
  [ok] TP reachable at ATR floor within window
       ↳ days to TP at 2.5%/day ≈ 1.2


Applied to the candidates that cleared the screen, the sizing rule keeps roughly constant dollar risk per trade: a calmer coin earns a larger clip, a livelier one a smaller clip, and nothing is allowed past the per-position cap or below the venue minimum. At this account size the rule pushes toward a few larger positions rather than many tiny clips that waste edge on fees.

In [25]:

cands = screen[screen["pass"]].copy()
if len(cands):
    plans = cands["atr_pct"].apply(lambda a: pd.Series(position_plan(a)))
    sized = pd.concat([cands[["symbol","atr_pct"]].reset_index(drop=True),
                       plans.reset_index(drop=True)[["stop_pct","position_usdt","floored_at_min"]]], axis=1)
    sized = sized.sort_values("position_usdt", ascending=False).reset_index(drop=True)
    display(sized)
    fig = go.Figure(go.Bar(x=sized["symbol"].str.replace("/USDT",""), y=sized["position_usdt"],
                           marker_color="#0B3D66",
                           text=[f"ATR {a:.1f}%" for a in sized["atr_pct"]], textposition="outside"))
    fig.add_hline(y=CONFIG["account_usdt"]*CONFIG["max_position_pct"]/100,
                  line=dict(color="#B22222", dash="dash"),
                  annotation_text=f"position cap ({CONFIG['max_position_pct']}% of acct)")
    fig.add_hline(y=CONFIG["min_notional_usdt"], line=dict(color="#888", dash="dot"),
                  annotation_text="min notional")
    fig.update_layout(title="ATR-scaled position sizing — constant $ risk (higher ATR → smaller clip)",
                      template="plotly_white", height=360, yaxis_title="position size (USDT)",
                      margin=dict(l=50, r=20, t=50, b=40))
    show(fig)
else:
    print("No candidates cleared the screen on this run — widen the universe or revisit the band.")

,symbol,atr_pct,stop_pct,position_usdt,floored_at_min
0,BTC/USDT,3.327,4.990,150.29,False
1,BNB/USDT,3.500,5.250,142.86,False
2,ETH/USDT,4.724,7.086,105.84,False
3,XRP/USDT,5.077,7.615,98.48,False
4,SOL/USDT,5.629,8.444,88.83,False
5,AVAX/USDT,6.897,10.346,72.50,False
6,TAO/USDT,9.743,14.614,51.32,False
7,NEAR/USDT,10.151,15.226,49.26,False
8,ZEC/USDT,11.183,16.774,44.71,False


### Experiment Log & Frozen Harness

The Karpathy lift, adapted: the yardstick is **frozen and operator-owned**, and every walk-forward run appends **one line** — params tried, out-of-sample after-fee result, kept or discarded — to a log the operator reviews weekly. The model never writes to either. This notebook only scaffolds them; the walk-forward loop that fills the log is the next build, and it stays the human's review surface, not something automated away.

In [26]:

# Frozen evaluation harness — versioned, identical across every comparison. 🔒
HARNESS = dict(
    version            = "harness-v1",
    oos_window_days    = 90,
    train_window_days  = 365,
    embargo_days       = 20,                       # straddles the train/test cut
    regime_set         = ["bull", "bear", "sideways"],
    fee_assumption_pct = CONFIG["round_trip_fee_pct"],
    metric             = "oos_after_fee_expectancy",
)

def log_experiment(params: dict, oos_result: dict, kept: bool, why: str,
                   path=OUTPUTS / "experiment_log.csv"):
    """Append one line per walk-forward run. The artefact the operator wakes to."""
    row = {"ts_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "harness": HARNESS["version"], "kept": bool(kept), "why": why,
           **{f"p_{k}": v for k, v in params.items()},
           **{f"oos_{k}": v for k, v in oos_result.items()}}
    df = pd.DataFrame([row])
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)
    return path

# Demonstration entry (replace with real walk-forward output once that loop exists).
p = log_experiment(
    params=dict(edge_floor_pct=CONFIG["edge_floor_pct"], take_profit_pct=CONFIG["take_profit_pct"],
                atr_floor_pct=CONFIG["atr_floor_pct"], threshold=2),
    oos_result=dict(after_fee_expectancy_pct=-0.4, win_rate=0.41, n_trades=37, vs_buy_hold_pct=+1.2),
    kept=False, why="no demonstrated OOS edge after fees (matches standing NO-GO)")
print("logged ->", p)
pd.read_csv(p).tail(3)

logged -> outputs/experiment_log.csv


,ts_utc,harness,kept,why,p_edge_floor_pct,p_take_profit_pct,p_atr_floor_pct,p_threshold,oos_after_fee_expectancy_pct,oos_win_rate,oos_n_trades,oos_vs_buy_hold_pct
0,2026-06-20T01:00:53+00:00,harness-v1,False,no demonstrated OOS edge after fees (matches s...,1.5,3.0,2.5,2,-0.4,0.41,37,1.2
1,2026-06-20T01:02:23+00:00,harness-v1,False,no demonstrated OOS edge after fees (matches s...,1.5,3.0,2.5,2,-0.4,0.41,37,1.2


### Outputs

The dated candidate table and the experiment log are written under `outputs/`, joining the existing run artefacts there. The candidate table is the deliverable the screen produces each week; re-run weekly and the universe refreshes itself.

In [27]:

print("Artefacts written this run:")
for f in sorted(OUTPUTS.glob("Candidates_*.csv")) + [OUTPUTS / "experiment_log.csv"]:
    if f.exists():
        print(f"  {f}  ({f.stat().st_size} bytes)")
print("\nReminder: this notebook screens and sizes only. No orders are placed; "
      "LIVE_TRADING stays 'false' in inputs/config.py and is never read here.")

Artefacts written this run:
  outputs/Candidates_20260620.csv  (1983 bytes)
  outputs/experiment_log.csv  (432 bytes)

Reminder: this notebook screens and sizes only. No orders are placed; LIVE_TRADING stays 'false' in inputs/config.py and is never read here.
